### Project Overview
This notebook implements a full end-to-end pipeline for the CSIRO Image2Biomass prediction task. 
https://www.kaggle.com/competitions/csiro-biomass/overview

The goal is to predict five biomass-related targets from plot images and associated metadata, with a competition metric based on Kaggle-style weighted R² over all targets.

raw CSV + JPEGs → processed folds → baselines → hybrid teacher → distilled student


In [39]:
# ==================== 0. Setup Reproducibility & Config ====================
import os, json, random, platform
from dataclasses import dataclass, asdict
from pathlib import Path

# Make TensorFlow as deterministic as possible (given hardware / ops)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1" 
os.environ["PYTHONHASHSEED"] = str(42) 

import tensorflow as tf

tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
import cv2 # OpenCV (Open Source Computer Vision Library)
import sklearn
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedGroupKFold
from datetime import datetime

# Log library versions for reproducibility
versions = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "keras": tf.keras.__version__,
}
print("versions:", versions)

@dataclass
class CFG:
    """Global configuration for the CSIRO Image2Biomass project.

    This dataclass centralizes all tunable knobs and file paths so that
    the rest of the notebook can read configuration from a single object.

    Attributes:
        project_title: Human-readable project title.
        seed: Global random seed used for NumPy, Python, and TensorFlow.
        image_size_h: Height of input images in pixels.
        image_size_w: Width of input images in pixels.
        n_folds: Number of cross-validation folds.
        batch_size: Batch size for training and evaluation.
        raw_csv: Relative path to the original long-format training CSV.
        image_root: Root directory for image files referenced in the CSV.
        artifacts_dir: Directory where processed CSVs, configs, and models
            are saved.
        figs_dir: Directory where figures and diagnostic plots are saved.
        use_log1p: Whether to use log1p-transformed labels during training.
    """
    project_title: str = "CSIRO Image2Biomass Prediction"
    seed: int = 42
    image_size_h: int = 224
    image_size_w: int = 448 
    n_folds: int = 5
    batch_size: int = 8 # batch_size=8 for 357 images can be stable and better generalization
    raw_csv: str = "../csiro-biomass/train.csv"
    image_root: str = "../csiro-biomass"
    artifacts_dir: str = "../artifacts/"
    figs_dir: str = "../figs"
    use_log1p: bool = False

CFG = CFG()
Path(CFG.artifacts_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.figs_dir).mkdir(parents=True, exist_ok=True)

# ---------- Targets ----------
CORE_TARGETS = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g"]  
RAW_TARGETS  = CORE_TARGETS + ["GDM_g","Dry_Total_g"]       


# ---------- Reproducibility ----------
def set_seed(seed=42):
    """Set global random seed for Python, NumPy, and TensorFlow"""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(CFG.seed)

versions: {'python': '3.11.14', 'tensorflow': '2.17.1', 'numpy': '1.26.4', 'pandas': '2.3.3', 'sklearn': '1.7.2', 'keras': '3.12.0'}


In [40]:
# ==================== 1. read raw data ====================
df_raw = pd.read_csv(CFG.raw_csv)
display(df_raw.head(5))
# add tensor columns: abs_image_path & image_id
df_raw["abs_image_path"] = df_raw["image_path"].apply(lambda p: str(Path(CFG.image_root) / p))
df_raw["image_id"] = df_raw["abs_image_path"].map(lambda p: Path(p).stem)


,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
1,ID1011485656__Dry_Dead_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
2,ID1011485656__Dry_Green_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Green_g,16.2751
3,ID1011485656__Dry_Total_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Total_g,48.2735
4,ID1011485656__GDM_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,GDM_g,16.2750


In [41]:
# ==================== 2. Pivot ====================
def build_pivot_df(df):
    """convert long-format biomass table to a wide per-image table."""
    
    # 1) create wide table which has one row per image with all 5 targets as columns
    wide = (
        df.pivot_table(index="image_id", # → each img is a row
                       columns="target_name", # → the target columns to create 
                       values="target",  # → the values to fill the table
                       aggfunc="first") # → in case of duplicates, take the first
          .reset_index() # → add the index column, so image_id becomes a column again
    )
    
    # 2)target columns check ensure all present
    for col in RAW_TARGETS:
        if col not in wide.columns:
            wide[col] = np.nan # add missing value target columns with NaN values
    wide = wide[["image_id"] + RAW_TARGETS]

    # Join metadata: from original df extract unique rows
    meta_cols = ["image_id", "abs_image_path", "Sampling_Date", "State", "Species", "Pre_GSHH_NDVI", "Height_Ave_cm"]
    meta = df[meta_cols].drop_duplicates("image_id") # drop duplicate rows based on image_id
    out = wide.merge(meta, on="image_id", how="left") # merge: use wide as base, left join meta

    cols =  meta_cols + RAW_TARGETS # reorder columns: metadata → targets(features first, then targets)
    return out[cols]

# create and save (CSV)
df_pivot = build_pivot_df(df_raw)

out_dir = Path(CFG.artifacts_dir)
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "train_wide.csv"
df_pivot.to_csv(csv_path, index=False)

print("Saved:", csv_path)
print("Pivot shape:", df_pivot.shape)


Saved: ../artifacts/train_wide.csv
Pivot shape: (357, 12)


In [42]:

# ==================== 3. Date features (update df_pivot)====================
from datetime import datetime
# 1) Date features
df_pivot["Sampling_Date"] = pd.to_datetime(df_pivot["Sampling_Date"], format="%Y/%m/%d", errors="coerce")
df_pivot["year"] = df_pivot["Sampling_Date"].dt.year
df_pivot["month"] = df_pivot["Sampling_Date"].dt.month
df_pivot["day_of_year"] = df_pivot["Sampling_Date"].dt.dayofyear

# 2) cyclical encodings
df_pivot['month_sin'] = np.sin(2 * np.pi * df_pivot['month'] / 12)
df_pivot['month_cos'] = np.cos(2 * np.pi * df_pivot['month'] / 12)
df_pivot['day_of_year_sin'] = np.sin(2 * np.pi * df_pivot['day_of_year'] / 365.25)
df_pivot['day_of_year_cos'] = np.cos(2 * np.pi * df_pivot['day_of_year'] / 365.25)

# 3) Categorical encoding (for tabular model)
# Convert "State" and "Species" features into numbers and save the mapping relationship for use during inference!
state_cat = df_pivot["State"].astype("category")    # 1. convert to category dtype
species_cat = df_pivot["Species"].astype("category")
df_pivot["State_encoded"] = state_cat.cat.codes     # 2. extract integer code as new column
df_pivot["Species_encoded"] = species_cat.cat.codes
STATE_MAP = dict(enumerate(state_cat.cat.categories)) # 3. create mapping dicts → int to category
SPECIES_MAP = dict(enumerate(species_cat.cat.categories))

# Reorder columns: metadata → targets
feature_cols = [col for col in df_pivot.columns if col not in RAW_TARGETS]
df_pivot = df_pivot[feature_cols + RAW_TARGETS]  # reorder columns: features first, then targets


In [43]:
# ==================== 4. StratifiedKFold by Species + Dry_Total_g ====================

from sklearn.model_selection import StratifiedKFold

# 1) Build stratification labels: Species + binned Dry_Total_g
y_for_bins = df_pivot["Dry_Total_g"]
if y_for_bins.isna().any():
    y_for_bins = (
        df_pivot["Dry_Green_g"] 
        + df_pivot["Dry_Dead_g"] 
        + df_pivot["Dry_Clover_g"]
    )

# Bin Dry_Total_g into 5 quantile-based buckets
bins = pd.qcut(y_for_bins, q=5, labels=False, duplicates="drop")

# Stratification label = species name + biomass bin index
strata = df_pivot["Species"].astype(str) + "_" + bins.astype(str)

# 2) StratifiedKFold: split into 5 folds, shuffled, stratified by `strata`
#    This is what makes each fold about 357 / 5 ≈ 71 samples.
skf = StratifiedKFold(
    n_splits=CFG.n_folds,   # decide how many folds → ~71 samples per fold
    shuffle=True,
    random_state=546195, # 7499
)

folds = np.full(len(df_pivot), -1)

for fold, (_, val_idx) in enumerate(skf.split(df_pivot, y=strata)):
    folds[val_idx] = fold

df_pivot["fold"] = folds.astype(int)

# 3) Sanity check: no species should appear in only a single fold
violations = []
species_counts = df_pivot["Species"].value_counts()

for sp, total in species_counts.items():
    per_fold = df_pivot[df_pivot["Species"] == sp]["fold"].value_counts()
    for fold_id, cnt in per_fold.items():
        if cnt == total:  # if this species is entirely contained in one fold
            violations.append((sp, fold_id, int(cnt), int(total)))

if violations:
    print("[Warn] Some species only appear in a single fold:")
    for sp, f, c, tot in violations:
        print(f"  - Species={sp}, fold={f}, count={c}/{tot}")
    # If you want to enforce this as a hard constraint, you can replace the
    # prints above with: `assert not violations`

# 4) Report: distribution of Dry_Total_g in each fold (original scale)
def fold_report(df):
    tgts = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"]
    return df.groupby("fold")[tgts].agg(["mean","std","min","max","count"]).round(2)

print("\n=== Fold report (on original scale, before any log transform) ===")
display(fold_report(df_pivot))

print("\n=== Fold sizes ===")
for fold_num in range(CFG.n_folds):
    print(f"Fold {fold_num}: {(df_pivot['fold'] == fold_num).sum()} samples")

# 5) save
processed_path = Path(CFG.artifacts_dir) / "train_processed.csv"
df_pivot.to_csv(processed_path, index=False)
print("Saved processed with folds to:", processed_path)


=== Fold report (on original scale, before any log transform) ===


/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Dry_Green_g                            Dry_Dead_g                     \
            mean    std   min     max count       mean    std  min    max   
fold                                                                        
0          25.67  22.78  0.00  129.34    72      13.57  15.35  0.0  83.84   
1          25.85  24.08  0.00  116.75    72      11.62  12.07  0.0  50.23   
2          28.72  29.14  0.00  119.60    71      10.11  11.21  0.0  53.26   
3          24.88  22.17  0.31   83.93    71      13.34  11.78  0.0  46.07   
4          28.04  28.56  0.00  157.98    71      11.57  11.09  0.0  52.93   

            ...  GDM_g                            Dry_Total_g               \
     count  ...   mean    std   min     max count        mean    std   min   
fold        ...                                                              
0       72  ...  31.24  21.65  1.40  129.34    72       44.80  28.16  6.30   
1       72  ...  33.00  24.31  3.78  116.75    72       44.63  24.36  9.20   
2       71  ...  36.44  27.72  1.80  119.60    71       46.54  31.75  2.48   
3       71  ...  32.14  22.23  2.40   96.76    71       45.49  24.89  4.30   
4       71  ...  33.59  28.45  1.04  157.98    71       45.15  30.77  1.04   

                   
        max count  
fold               
0     157.9    72  
1     120.4    72  
2     166.1    71  
3     119.1    71  
4     185.7    71  

[5 rows x 25 columns]


=== Fold sizes ===
Fold 0: 72 samples
Fold 1: 72 samples
Fold 2: 71 samples
Fold 3: 71 samples
Fold 4: 71 samples
Saved processed with folds to: ../artifacts/train_processed.csv


In [44]:
# ==================== 5. Label transform (log1p or gram) & Save processing_config ====================

# 1) Regardless of whether the current one is used or not, calculate the *_log column first (only once)
for t in RAW_TARGETS:
    neg_ct = (df_pivot[t] < 0).sum()
    if neg_ct > 0:
        print(f"[Warn] {t} has {neg_ct} negatives; clipping to 0 before log1p.")
        df_pivot.loc[df_pivot[t] < 0, t] = 0.0
        
for t in RAW_TARGETS:
    log_col = f"{t}_log"
    if log_col not in df_pivot.columns:
        df_pivot[log_col] = np.log1p(df_pivot[t].astype("float32"))
        
# ====================        

# 2) Define a helper to switch label mode (log1p vs gram)
def set_label_mode(use_log1p: bool):
    """
    Switch the target columns used during training:
        use_log1p=True → Use the *_log columns (log1p)
        use_log1p=False → Use the original gram columns 
    """
    global TRAIN_TARGETS, label_transform
    
    if not hasattr(CFG, 'use_log1p'):
        CFG.use_log1p = False
    
    CFG.use_log1p = use_log1p

    if use_log1p:
        TRAIN_TARGETS = [f"{t}_log" for t in RAW_TARGETS]
        label_transform = "log1p"
    else:
        TRAIN_TARGETS = RAW_TARGETS
        label_transform = "identity" # 'identity' means no transform / raw grams

    print(f"[set_label_mode] use_log1p={use_log1p} → TRAIN_TARGETS =", TRAIN_TARGETS)
    print(f"[init] fallback label_transform={label_transform}")
    
# 3) Set a default mode (e.g., gram) for preprocessing/config
set_label_mode(False)  # default set to gram
# ====================

# Overwrite save (contains both raw and *_log)
processed_path = Path(CFG.artifacts_dir) / "train_processed.csv"
df_pivot.to_csv(processed_path, index=False)
print("Saved (with log columns):", processed_path)


# Assemble and save processing_config.json
processing_cfg = {
    "project_title": CFG.project_title,
    "seed": CFG.seed,
    "image_size_h": CFG.image_size_h,
    "image_size_w": CFG.image_size_w,
    "n_folds": CFG.n_folds,
    "batch_size": CFG.batch_size,
    "use_log1p": CFG.use_log1p,  # will reflect the last mode you set
    "label_transform": label_transform,
    "train_target_cols": TRAIN_TARGETS,         # train reads these columns
    "train_target_cols_raw": CORE_TARGETS,      # corresponding physical meanings
    "all_targets_raw": RAW_TARGETS,
    "artifacts_dir": CFG.artifacts_dir,
    "image_root": CFG.image_root,
    "n_samples": int(len(df_pivot)),
    "n_unique_images": int(df_pivot["image_id"].nunique()),
    "State_map": STATE_MAP,
    "Species_map": SPECIES_MAP,
}

# save json
proc_cfg_path = Path(CFG.artifacts_dir) / "processing_config.json"
proc_cfg_path.parent.mkdir(parents=True, exist_ok=True)
with open(proc_cfg_path, "w", encoding="utf-8") as f:
    json.dump(processing_cfg, f, indent=2, ensure_ascii=False)

print("Saved processing config:", proc_cfg_path)

[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
[init] fallback label_transform=identity
Saved (with log columns): ../artifacts/train_processed.csv
Saved processing config: ../artifacts/processing_config.json


In [45]:
# ==================== 6. Config & Processed Data Saving ====================

from pathlib import Path
import json
from dataclasses import asdict
import numpy as np

# 1) Define the list of metadata features to be used
METADATA_FEATURES = [
    "Pre_GSHH_NDVI",      # Pre-season NDVI from satellite
    "Height_Ave_cm",      # Average plant height in cm
    "State_encoded",      # One-hot or label-encoded growth state
    "Species_encoded",    # Encoded plant species
    "year",               # Year of observation
    'month_sin',          # Cyclic encoding of month (sin)
    'month_cos',          # Cyclic encoding of month (cos)
    'day_of_year_sin',    # Cyclic encoding of day-of-year (sin)
    'day_of_year_cos',    # Cyclic encoding of day-of-year (cos)
]

# 2) Automatically separate categorical and continuous metadata columns
META_COLS_CAT  = [col for col in METADATA_FEATURES if 'encoded' in col]   # Categorical (already encoded)
META_COLS_CONT = [col for col in METADATA_FEATURES if col not in META_COLS_CAT]  # Continuous

# 3) Apply Z-score standardization to continuous metadata features
for col in META_COLS_CONT:
    mean_val = df_pivot[col].mean()
    std_val  = df_pivot[col].std()
    
    # Avoid division by zero or NaN
    if std_val == 0 or np.isnan(std_val):
        std_val = 1.0
    
    df_pivot[col] = (df_pivot[col] - mean_val) / std_val

print(f"Standardized continuous metadata features: {META_COLS_CONT}")
print(f"Categorical (encoded) metadata features: {META_COLS_CAT}")

# 4) Build configuration dictionary and save processed data
config_dict = {
    **asdict(CFG),                    # Include all settings from CFG dataclass
    "targets": RAW_TARGETS,    
    "metadata_features": METADATA_FEATURES,
    "n_samples": len(df_pivot),
    "n_unique_images": df_pivot["image_id"].nunique(),
    "n_metadata_features": len(METADATA_FEATURES),
    "n_categorical_meta": len(META_COLS_CAT),
    "n_continuous_meta": len(META_COLS_CONT),
}

# Pretty-print the final config
print("\nFinal Configuration:")
print(json.dumps(config_dict, indent=4, default=str))

# 5) Save the processed training dataframe (with folds, normalized features, etc.)
processed_path = Path(CFG.artifacts_dir) / "train_processed.csv"
df_pivot.to_csv(processed_path, index=False)



Standardized continuous metadata features: ['Pre_GSHH_NDVI', 'Height_Ave_cm', 'year', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos']
Categorical (encoded) metadata features: ['State_encoded', 'Species_encoded']

Final Configuration:
{
    "project_title": "CSIRO Image2Biomass Prediction",
    "seed": 42,
    "image_size_h": 224,
    "image_size_w": 448,
    "n_folds": 5,
    "batch_size": 8,
    "raw_csv": "../csiro-biomass/train.csv",
    "image_root": "../csiro-biomass",
    "artifacts_dir": "../artifacts/",
    "figs_dir": "../figs",
    "use_log1p": false,
    "targets": [
        "Dry_Green_g",
        "Dry_Dead_g",
        "Dry_Clover_g",
        "GDM_g",
        "Dry_Total_g"
    ],
    "metadata_features": [
        "Pre_GSHH_NDVI",
        "Height_Ave_cm",
        "State_encoded",
        "Species_encoded",
        "year",
        "month_sin",
        "month_cos",
        "day_of_year_sin",
        "day_of_year_cos"
    ],
    "n_samples": 357,
    "n_unique_i

## 2. images

In [46]:

import tensorflow as tf
from tensorflow.keras import layers

# ==================== 7. Augmentation primitives ====================

class ImageAugmentor(layers.Layer):
    """
    Custom Keras Layer for random image augmentation.
    
    Supports both:
      - Rank-3 inputs: (H, W, C)   → single image
      - Rank-4 inputs: (N, H, W, C) → batch of images
      
    Designed to be used:
      - Directly inside a Keras model (as a layer)
      - Or in tf.data pipelines via .map(tfdata_augmentor)
      
    Only augments the image (x), leaves labels (y) untouched.
    """
    
    def __init__(self, 
                 color_prob=0.15,
                #  gray_prob=0.0,
                 noise_prob=0.2,
                 blur_prob=0.2,
                 flip_lr_prob=0.5,
                 flip_ud_prob=0.5,
                 rot90_prob=0.5,
                 **kwargs):
        super().__init__(**kwargs)
        
        # augmentation config: (name, probability, apply function)
        self.ops = [
            ('Hue +0.02', color_prob, 
             lambda x: tf.image.adjust_hue(x, 0.02)),
            ('Saturation x1.1', color_prob, 
             lambda x: tf.image.adjust_saturation(x, 1.1)),
            ('Brightness +0.1', color_prob, 
             lambda x: tf.image.adjust_brightness(x, 0.1)),
            ('Contrast x1.1', color_prob, 
             lambda x: tf.image.adjust_contrast(x, 1.1)),
            # ('Grayscale', gray_prob, 
            #  lambda x: tf.image.grayscale_to_rgb(tf.image.rgb_to_grayscale(x))),
            ('Noise σ=0.02', noise_prob, 
             lambda x: x + tf.random.normal(tf.shape(x), stddev=0.02)),
            ('BoxBlur 3x3', blur_prob, 
             lambda x: tf.nn.avg_pool2d(x, 3, 1, 'SAME')),
            ('Flip LR', flip_lr_prob, 
             tf.image.flip_left_right),
            ('Flip UD', flip_ud_prob, 
             tf.image.flip_up_down),
            # ('Rotate 90°', rot90_prob, 
            #  lambda x: tf.image.rot90(x, k=1)),
        ]


    def get_visualization_ops(self, include_prob=False):
        """
        Helper to visualize what augmentations are applied (useful for debugging/logging).
        """
        if include_prob:
            return [(f"{name} ({prob:.0%})", fn) for name, prob, fn in self.ops]
        return [(name, fn) for name, prob, fn in self.ops]
    
    def call(self, x, training=None):
        """
        Keras Layer: g(x) -> x_aug
        - training=True/None: apply random augmentations
        - training=False: return img unchanged
        """
        # Handle single image (H,W,C) → temporarily add batch dim
        is_single = x.shape.rank == 3
        if is_single:
            x = x[None]  # (H,W,C) -> (1,H,W,C)
            
        # Default behavior in Keras: training=None means apply augmentations
        if training is None:
            training = True 
            
        # If training is a Tensor (e.g., traced in tf.data), use tf.cond to control overall logic
        def apply_all_ops(img):
            for _, prob, fn in self.ops:
                if prob > 0:
                    img = tf.cond(
                        tf.random.uniform(()) < prob,
                        lambda: tf.clip_by_value(fn(img), 0., 1.),
                        lambda: img,
                    )
            return img

        #  when training is a Tensor
        if isinstance(training, tf.Tensor):
            x = tf.cond(
                tf.cast(training, tf.bool),
                lambda: apply_all_ops(x),
                lambda: x,
            )
        elif training:
            x = apply_all_ops(x)
            
        # Remove batch dim if input was a single image
        return x[0] if is_single else x

# Instantiate the augmentor layer (can be reused)
img_aug_layer = ImageAugmentor(name="augmentor")

# def tfdata_augmentor(x, y): # f(x, y) → (x_aug, y)
#     return img_aug_layer(x, training=True), y


def tfdata_augmentor(x, y):
    """
    Universal augmentation wrapper for tf.data pipelines.
    
    Supports two input formats:
      - Image-only: x is a Tensor (H,W,C) or (N,H,W,C)
      - Hybrid model: x is a dict with key 'image'
      
    Returns: (x_augmented, y) — labels unchanged
    """
    # Case 1: x = (image, metadata)
    if isinstance(x, dict):
        x = dict(x)  # shallow copy to avoid mutating original
        x["image"] = img_aug_layer(x["image"], training=True)
        return x, y
    # Case 2: x = image only
    else:
        x_aug = img_aug_layer(x, training=True)
        return x_aug, y


def tfdata_clip(x, y): # Numerical Safety / Clamping
    """
    Universal clipping function (ensures pixel values stay in [0, 1]).
    Works with both image-only and dict-style inputs.
    """
    if isinstance(x, dict):
        x = dict(x)
        x["image"] = tf.clip_by_value(x["image"], 0.0, 1.0)
        return x, y
    else:
        return tf.clip_by_value(x, 0.0, 1.0), y


In [47]:
# ==================== 8. BiomassImgDataModule  ====================

class BiomassImgDataModule:
    def __init__(self, 
                 fold: int = 3, 
                 target_list: list = None,
                 include_meta: bool = False):  # <--- 1. tabular switch: Teacher-True,  Student/Baseline-False
        """
        include_meta=True: return ({'image': img, 'meta_cont':..., 'meta_cat':...}, y) → to Hybrid Teacher
        
        → **Input (x)**: A dictionary containing the image and all tabular metadata.
        → **Output (y)**: The 5 core biomass targets.
        
        ---------------------------------------------------------
        
        include_meta=False: return (img, y) → to Image-Only Student/Baseline
        
        → **Input (x)**: A tensor containing only the image (One image).
        → **Output (y)**: The 5 core biomass targets.
        
        **Summary:** Each record (One sample) corresponds to 5 targets.
        """
        self.fold = fold
        self.include_meta = include_meta       # <--- 2. include tabular
        
        if target_list is None:
            target_list = TRAIN_TARGETS
        self.target_list = target_list

        self.IMG_SIZE = (CFG.image_size_h, CFG.image_size_w)

        # --------------------- Load processed data ---------------------
        pivot_path = out_dir / "train_processed.csv"
        assert pivot_path.exists(), f"Missing: {pivot_path}"
        df = pd.read_csv(pivot_path)

        # Drop rows missing any target (critical for regression)
        df = df.dropna(subset=self.target_list).reset_index(drop=True)

        self.df = df
        self.y = df[self.target_list].astype("float32").values
        
        # --------------------- Metadata preprocessing (if used) ---------------------
        if self.include_meta:
            # 1. 
            self.meta_cols_cat  = META_COLS_CAT
            self.meta_cols_cont = META_COLS_CONT

            # Fill NaN in continuous metadata (common & safe default)
            self.df[self.meta_cols_cont] = self.df[self.meta_cols_cont].fillna(0.0)
            

    def load_img(self, path: str):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        img = cv2.resize(
            img,
            (self.IMG_SIZE[1], self.IMG_SIZE[0]), 
            interpolation=cv2.INTER_AREA
        )
        return img.astype("float32") / 255.0

    def make_arrays(self):
        # imgs → N x H x W x C
        imgs = np.stack([self.load_img(p) for p in self.df["abs_image_path"]])
        return imgs

    def make_datasets(self):
        imgs = self.make_arrays()
        y = self.y # y → N x T

        val_mask = (self.df["fold"] == self.fold).values
        tr_idx = np.where(~val_mask)[0]
        va_idx = np.where(val_mask)[0]

        print(f"Fold {self.fold} | Target: {self.target_list}")
        print(f"  Train: {len(tr_idx)} images | Val: {len(va_idx)} images")

        # # tf.data datasets
        # train_ds = tf.data.Dataset.from_tensor_slices((imgs[tr_idx], y[tr_idx]))
        # val_ds   = tf.data.Dataset.from_tensor_slices((imgs[va_idx], y[va_idx]))
        
        # ==========================================
        # Core Logic: Choose input format based on whether metadata is used
        # ==========================================
        if self.include_meta:
            # Mode A: Hybrid (Teacher) -> Returns a dictionary of inputs
            # Useful for models that accept multiple inputs (image + tabular metadata)
            meta_cont = self.df[self.meta_cols_cont].astype("float32").values # (N, 7)
            meta_cat  = self.df[self.meta_cols_cat].astype("int32").values # (N, 2)
            
            # Dataset yields (dict_of_inputs, labels)
            train_inputs = {
                "image":     imgs[tr_idx],
                "meta_cont": meta_cont[tr_idx],
                "meta_cat":  meta_cat[tr_idx],
            }
            val_inputs = {
                "image":     imgs[va_idx],
                "meta_cont": meta_cont[va_idx],
                "meta_cat":  meta_cat[va_idx],
            }

            train_ds = tf.data.Dataset.from_tensor_slices((train_inputs, y[tr_idx]))
            val_ds   = tf.data.Dataset.from_tensor_slices((val_inputs,   y[va_idx]))

        else:
            # Mode B: Image Only (Baseline / Student) -> Returns Tensor
            # Simpler and faster - perfect for pure CNNs or knowledge distillation students
            train_ds = tf.data.Dataset.from_tensor_slices((imgs[tr_idx], y[tr_idx]))
            val_ds   = tf.data.Dataset.from_tensor_slices((imgs[va_idx], y[va_idx]))


        # ==========================================
        # Pipeline (for all model)
        # ==========================================
        print(f"Fold {self.fold} | Meta={self.include_meta} | Train:{len(tr_idx)} Val:{len(va_idx)}")
        
        # Create Image Data Pipeline
        train_ds = (
            train_ds
            .map(tfdata_augmentor, num_parallel_calls=1)# x passed to aug function, preprocessing + Augmentation
            .shuffle(1024, seed=CFG.seed, reshuffle_each_iteration=False) # Shuffle order
            .batch(CFG.batch_size)  # batching
            .prefetch(1) # prefetch for performance(speed up training, important!)
        )

        val_ds = (
            val_ds
            .map(tfdata_clip, num_parallel_calls=1)
            .batch(CFG.batch_size)
            .prefetch(1)
        )

        return train_ds, val_ds


In [48]:
dm = BiomassImgDataModule(fold=1)
imgs = dm.make_arrays()
print(imgs.shape)   # should be (N, 112, 224, 3)

(357, 224, 448, 3)


## 3. cnn model

In [49]:

# ==================== 9. Weighted R² metric  ====================
import numpy as np
import pandas as pd

COMP_WEIGHTS = {
   "Dry_Green_g": 0.1,
   "Dry_Dead_g": 0.1,
   "Dry_Clover_g": 0.1,
   "GDM_g": 0.2,
   "Dry_Total_g": 0.5,
}

def get_loss_weights(target_names):
    """
    Given a list of training target column names (possibly *_log),
    return the per-target weights aligned with COMP_WEIGHTS.

    Example:
        target_names = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g"]
        or         = ["Dry_Green_g_log", ...] in log1p mode.

    Returns:
        list[float]: weights in the same order as target_names.
    """
    base_names = [name.replace("_log", "") for name in target_names]
    return [COMP_WEIGHTS[n] for n in base_names]


# LOSS_WEIGHTS_RAW = [COMP_WEIGHTS[t] for t in RAW_TARGETS]

# get_loss_weights(TRAIN_TARGETS) = [0.1,0.1,0.1,0.2,0.5]
# Loss weights for the three core competition targets
# loss_weights = get_loss_weights(target_names)


def weighted_r2_kaggle(df_true, df_pred):
    """
    df_true, df_pred: DataFrame, with columns:
      ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']
    All values are on the ORIGINAL scale (grams), no logs.
    """

    cols = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"]

    y_list, yp_list, w_list = [], [], []

    for col in cols:
        y  = df_true[col].astype("float64").values
        yp = df_pred[col].astype("float64").values
        w  = np.full_like(y, fill_value=COMP_WEIGHTS[col], dtype="float64")

        y_list.append(y)
        yp_list.append(yp)
        w_list.append(w)

    y_all  = np.concatenate(y_list)      # all (image, target) in one array
    yp_all = np.concatenate(yp_list)
    w_all  = np.concatenate(w_list)

    # Global weighted average
    y_bar_w = np.sum(w_all * y_all) / np.sum(w_all)

    ss_res = np.sum(w_all * (y_all - yp_all)**2)
    ss_tot = np.sum(w_all * (y_all - y_bar_w)**2)

    r2w = 1.0 - ss_res / (ss_tot + 1e-12)
    return float(r2w)
      

def weighted_mse_loss(weights, max_err=None):
    """Create a weighted MSE loss function.

    Args:
        weights: 1D sequence of per-target weights.
        max_err: Optional max absolute error for clipping. If None, no clipping.

    Returns:
        Callable: Loss function `loss_fn(y_true, y_pred)` for Keras.
    """
    w = tf.constant(weights, tf.float32)

    def loss_fn(y_true, y_pred):
        err = y_true - y_pred  # (B, T)
        if max_err is not None:
            # Clip the absolute value of individual errors to prevent explosion
            err = tf.clip_by_value(err, -max_err, max_err)
        err2 = tf.square(err)
        return tf.reduce_mean(err2 * w)
    return loss_fn

def eval_weighted_r2_in_gram(
        val_df: pd.DataFrame,
        y_pred_default: np.ndarray,
        model_name: str = "",
        fold: int | None = None):
    """
    Evaluate model predictions using Kaggle-style Weighted R² in gram space.

    Args:
        val_df (DataFrame): Ground-truth labels in gram units.
        y_pred_default (ndarray): Model outputs (log1p or gram), shape (N, 3).
        model_name (str, optional): Name for logging.
        fold (int, optional): Fold index for logging.

    Returns:
        tuple: (weighted_r2, pred_df, true_df)
    """
    # 1) Convert model outputs to gram space based on the current label mode. log1p → gram /  gram
    if CFG.use_log1p:
        # default outputs are log1p → convert back to grams
        y_pred_gram = np.expm1(y_pred_default)
    else:
        # default outputs are already in grams
        y_pred_gram = y_pred_default
        
    n_out = y_pred_gram.shape[1]
    
    # 2) Build prediction DataFrame depending on output dimension. # pred_df
    if n_out == 3:
         # Old behavior: model outputs 3 core targets, we derive GDM & Dry_Total
        # Extract the three core targets
        dg_p = y_pred_gram[:, 0]
        dd_p = y_pred_gram[:, 1]
        dc_p = y_pred_gram[:, 2]

        # Construct prediction DataFrame (including derived GDM / Dry_Total)
        pred_df = pd.DataFrame({
            "Dry_Green_g":  dg_p,
            "Dry_Dead_g":   dd_p,
            "Dry_Clover_g": dc_p,
        })
        pred_df["GDM_g"]       = pred_df["Dry_Green_g"] + pred_df["Dry_Clover_g"]
        pred_df["Dry_Total_g"] = pred_df["Dry_Green_g"] + pred_df["Dry_Dead_g"] + pred_df["Dry_Clover_g"]
        
    elif n_out == 5:
        # New behavior: model directly outputs 5 targets in order RAW_TARGETS
        pred_df = pd.DataFrame(
            y_pred_gram,
            columns=["Dry_Green_g",
                     "Dry_Dead_g",
                     "Dry_Clover_g",
                     "GDM_g",
                     "Dry_Total_g"]
        )
    else:
        raise ValueError(f"Expected model to output 3 or 5 targets, got shape {y_pred_gram.shape}")


    # 3) Ground truth (always in gram space)
    true_df = val_df[[
        "Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"
    ]].copy()

    # 4) Compute Weighted R²
    w_r2 = weighted_r2_kaggle(true_df, pred_df)

    tag = f"[{model_name}] " if model_name else ""
    if fold is not None:
        print(f"{tag}Fold {fold} Kaggle-style weighted R2 = {w_r2:.4f}")
    else:
        print(f"{tag}Kaggle-style weighted R2 = {w_r2:.4f}")

    return w_r2, pred_df, true_df

In [50]:
# ==================== 10. backbone ====================
# this model, it uses ONLY pure image data — no tabular features at all!

from tensorflow.keras import layers, models, optimizers, callbacks
def make_mae_metrics(target_names):
    """
    Create overall MAE metric + per-target MAE metrics for Keras model compilation.
    
    Supports two scenarios:
    - Normal training: y_true shape = (B, T)
    - Distillation training: y_true shape = (B, 2T), 
      where the first T dimensions are ground truth labels,
      and the latter T dimensions are teacher predictions.
    """
    n_targets = len(target_names)

    def mae_all(y_true, y_pred):
        # Only evaluate on the ground truth part (first half)
        y_true_core = y_true[:, :n_targets]  
        return tf.reduce_mean(tf.abs(y_true_core - y_pred))

    mae_all.__name__ = "mae_all"
    metric_list = [mae_all]

    def make_fn(idx, name):
        def mae_i(y_true, y_pred):
            y_true_core = y_true[:, :n_targets]
            return tf.reduce_mean(tf.abs(y_true_core[:, idx] - y_pred[:, idx]))
        mae_i.__name__ = name
        return mae_i

    for i, tname in enumerate(target_names):
        metric_list.append(make_fn(i, f"mae_{tname}"))
    return metric_list


# === shared image backbone ===
def build_image_backbone(
    image_size_h: int = CFG.image_size_h,
    image_size_w: int = CFG.image_size_w,
    name: str = "ImageBackbone",
):
    """Build a basic CNN backbone for image feature extraction.

    Args:
        image_size_h: Input image height in pixels.
        image_size_w: Input image width in pixels.

    Returns:
        tf.keras.Model: CNN backbone mapping (H, W, 3) → (batch, 256) features.
    """
    
    img_shape = (image_size_h, image_size_w, 3)
    inputs = layers.Input(shape=img_shape)

    x = layers.Conv2D(16, 3, padding="same", activation=None,
                      kernel_initializer="he_normal")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)  # 112x224

    x = layers.Conv2D(32, 3, padding="same", activation=None,
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)  # 56x112

    x = layers.Conv2D(64, 3, padding="same", activation=None,
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)  # 28x56

    x = layers.Conv2D(256, 3, padding="same", activation=None,
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)   # -> (batch, 256)

    backbone = models.Model(inputs=inputs, outputs=x, name=name)
    return backbone
# ==================== 11A. BiomassConvBaseline (multi-target CNN regressor) ====================
def biomass_conv_baseline(
    image_size_h: int = CFG.image_size_h, 
    image_size_w: int = CFG.image_size_w, 
    target_names: list | None = None,
) -> tf.keras.Model:
    """
    Simple CNN baseline for multi-target biomass regression.
    Input : image (H,W,3)
    Output: len(target_names) regression values.
    """
    if target_names is None:
        target_names = TRAIN_TARGETS  # default to all targets
    n_targets = len(target_names)

    # Shared image backbone, as a layer for any model
    backbone = build_image_backbone(image_size_h, image_size_w, name="ImageBackbone")
    img_inputs = layers.Input(shape=(image_size_h, image_size_w, 3), name="image")
    x = backbone(img_inputs)

    # Head: regression part
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(
        n_targets,
        activation="softplus", # Softplus to ensure non-negative outputs
        name="biomass",
        dtype="float32",
    )(x)

    model = models.Model(
        inputs=img_inputs, 
        outputs=outputs,
        name="BiomassConvBaseline"
    )

    # Loss weights for all 5 competition targets
    loss_weights = get_loss_weights(target_names)

    model.compile(
        optimizer=optimizers.Adam(1e-3),
        loss=weighted_mse_loss(loss_weights), # Custom weighted MSE (assumed defined elsewhere)
        metrics=make_mae_metrics(target_names),
    )
    return model


In [51]:
# ==================== 11.0. refers: true Mean baseline ≈ 0.24  ====================
def mean_baseline_r2_for_fold(fold: int):
    """
    Compute weighted R² for a per-fold mean target baseline.

    For the given fold, this baseline predicts the mean of each target
    (computed on the training split) for all validation samples, then
    evaluates Kaggle-style weighted R².

    Args:
        fold: Fold index to use as validation split.

    Returns:
        float: Weighted R² score for this fold.
    """
    dm = BiomassImgDataModule(fold=fold, target_list=TRAIN_TARGETS)

    val_mask  = (dm.df["fold"] == fold).values
    train_mask = ~val_mask

    train_df = dm.df[train_mask].reset_index(drop=True)
    val_df   = dm.df[val_mask].reset_index(drop=True)

    # mean of train_df (gram)
    mean_df = train_df[["Dry_Green_g","Dry_Dead_g","Dry_Clover_g"]].mean()

    n_val = len(val_df)
    pred_df = pd.DataFrame({
        "Dry_Green_g":  np.full(n_val, mean_df["Dry_Green_g"]),
        "Dry_Dead_g":   np.full(n_val, mean_df["Dry_Dead_g"]),
        "Dry_Clover_g": np.full(n_val, mean_df["Dry_Clover_g"]),
    })
    pred_df["GDM_g"]       = pred_df["Dry_Green_g"] + pred_df["Dry_Clover_g"]
    pred_df["Dry_Total_g"] = pred_df["Dry_Green_g"] + pred_df["Dry_Dead_g"] + pred_df["Dry_Clover_g"]

    true_df = val_df[[
        "Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"
    ]].copy()

    r2 = weighted_r2_kaggle(true_df, pred_df)
    print(f"[Mean baseline] Fold {fold}: weighted R² = {r2:.4f}")
    return r2

mean_r2 = []
for f in range(CFG.n_folds):
    mean_r2.append(mean_baseline_r2_for_fold(f))

print("Mean-baseline R² per fold:", np.round(mean_r2, 3),
      "mean =", np.mean(mean_r2))


[Mean baseline] Fold 0: weighted R² = 0.2467
[Mean baseline] Fold 1: weighted R² = 0.2703
[Mean baseline] Fold 2: weighted R² = 0.2077
[Mean baseline] Fold 3: weighted R² = 0.2795
[Mean baseline] Fold 4: weighted R² = 0.2122
Mean-baseline R² per fold: [0.247 0.27  0.208 0.279 0.212] mean = 0.24327056587010784


3 target:
Mean-baseline R² per fold: [0.247 0.27  0.208 0.279 0.212] mean = 0.24327056587010784

In [52]:
# ==================== 11. one fold train generic model ====================

from tensorflow.keras import callbacks

def build_callbacks(monitor, mode):
    """Create standard training callbacks.

    Args:
        monitor: Metric name to monitor (e.g. "val_mae_all").
        mode: "min" or "max", passed to Keras callbacks.

    Returns:
        list: [EarlyStopping, ReduceLROnPlateau] callbacks.
    """
    rlr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        mode=mode,                
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    )
    es = callbacks.EarlyStopping(
        monitor=monitor,
        mode=mode,
        patience=5,
        restore_best_weights=True,
        verbose=1,
    )
    return [es, rlr] # callbacks_list, early_stopping


def build_val_inputs(
    dm: BiomassImgDataModule,
    fold: int,
    include_meta: bool,
):
    """Build validation inputs for R² evaluation.

    Args:
        dm: Data module holding processed DataFrame and image arrays.
        fold: Fold index to select validation rows.
        include_meta: If True, return dict(image + metadata); otherwise images only.

    Returns:
        tuple:
            - val_df (pd.DataFrame): Validation rows in gram space.
            - x_val: Validation inputs for model.predict (tensor or dict).
    """
    # 1) Select the current fold's val df (with 5 real targets in GRAM space)
    val_mask = (dm.df["fold"] == fold).values
    val_df   = dm.df[val_mask].reset_index(drop=True)

    # 2) prepare val img array
    imgs_all = dm.make_arrays()              # (N, H, W, 3)
    imgs_val = imgs_all[val_mask]            # (N_val, H, W, 3)

    # 3)Decide the structure of x_val based on include_meta
    # include_meta=False → image-only pipeline + x_val = imgs_val
    # include_meta=False → image-only pipeline + x_val = imgs_val
    # CFG.use_log1p control log/gram
    if include_meta:
        # Teacher: image + tabular dict
        meta_cont_all = dm.df[dm.meta_cols_cont].astype("float32").values
        meta_cat_all  = dm.df[dm.meta_cols_cat ].astype("int32").values

        meta_cont_val = meta_cont_all[val_mask]
        meta_cat_val  = meta_cat_all[val_mask]

        x_val = {
            "image":     imgs_val,
            "meta_cont": meta_cont_val,
            "meta_cat":  meta_cat_val,
        }
    else:
        # Baseline: image only
        x_val = imgs_val
    return val_df, x_val


def train_one_fold_generic(
    fold,
    *,
    include_meta: bool,
    model_fn,
    model_name: str,
    target_list=None,
    **model_kwargs, #  baseline_weights_path=baseline_ckpt pass to model_fn
):
    """
    Train one fold.

    If target_list is None, default to TRAIN_TARGETS.
    When using log1p, TRAIN_TARGETS should be the *_log names if CFG.use_log1p is True.
    Args:
        fold: which fold index to use as validation.
        target_list: list of target column names; default = TRAIN_TARGETS.
        model_fn: a function that builds and returns a compiled Keras model.
                  Signature: model_fn(image_size_h, image_size_w, target_names, **model_kwargs)
        model_kwargs: extra keyword args passed to model_fn (e.g. backbone_trainable_ratio).
    Returns:
        tuple: (best_val_mae, model, history, val_ds, w_r2)
    """
    
    if target_list is None:
        target_list = TRAIN_TARGETS

    print(f"\n===== Fold {fold} =====")
    print(f"[Baseline] use_log1p={CFG.use_log1p}, target_list={target_list}")

    # ---- Choose a fold for validation and train ----
    dm = BiomassImgDataModule(
        fold=fold, 
        target_list=target_list,
        include_meta=include_meta,   # <--- baseline exclude tabular,image only
    )
    train_ds, val_ds = dm.make_datasets()
  
    # ---- create model ----
    model = model_fn( # 9A.biomass_conv_baseline
        image_size_h = CFG.image_size_h, 
        image_size_w = CFG.image_size_w,
        target_names=target_list,
        **model_kwargs,
    )
  
    # ---- callbacks ----
    callbacks_list = build_callbacks(monitor="val_mae_all", mode="min")
    
    # ---- Train fold ----
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        callbacks=callbacks_list,
        verbose=1,
    )

    # ---- Evaluate fold ----
    #  Get best val_mae_all from 5 folds history
    best_val_mae = min(history.history["val_mae_all"])
    print(f"Fold {fold} best val_mae_all = {best_val_mae:.3f}")
    
    
    # =========  Weighted R² (eval ALWAYS in gram)  ========
    # build_val_inputs + eval_weighted_r2_in_gram
    
    # 1) Construct val_df + x_val (image-only or image+meta)
    val_df, x_val = build_val_inputs(
        dm=dm,
        fold=fold,
        include_meta=include_meta,
    )

    # 2) model predict (use gram to caculate R2), if its log1p, then convert to grams if needed
    y_pred_default= model.predict(x_val, batch_size=CFG.batch_size)
                        
    # 3) Calculate R² in gram
    w_r2, pred_df, true_df = eval_weighted_r2_in_gram(
        val_df=val_df,
        y_pred_default=y_pred_default,
        model_name=model_name,
        fold=fold,
    )
    # ========================================
    return best_val_mae, model, history, val_ds, w_r2,

In [53]:
import os
os.makedirs("./ckpt", exist_ok=True)
# wrappers 
def train_one_fold_baseline(
    fold,
    target_list=None,
    model_fn=biomass_conv_baseline,
    save_ckpt: bool = False,   # 👈 
    **model_kwargs,
):
    """
    Baseline CNN: image-only model (no metadata).
    return: best_val_mae, model, history, val_ds, w_r2
    option to save fold's weight to ./ckpt/
    """
    # Use the generic fold training pipeline
    best_val_mae, model, history, val_ds, w_r2 = train_one_fold_generic(
        fold=fold,
        model_fn=model_fn,
        include_meta=False,     # image-only baseline
        target_list=target_list,   # default = TRAIN_TARGETS via set_label_mode
        model_name=f"baseline_fold{fold}",
        **model_kwargs,
    )

    # Optionally save checkpoint for initializing student later
    if save_ckpt:
        mode = "log1p" if CFG.use_log1p else "gram"  
        ckpt_path = f"./ckpt/baseline_{mode}_fold{fold}.weights.h5"
        model.save_weights(ckpt_path)
        print(f"[Baseline] fold {fold} weights saved to {ckpt_path}")

    print(f"[Baseline] fold {fold} best val_mae_all = {best_val_mae:.3f}, "
          f"weighted R² = {w_r2:.3f}")
    return best_val_mae, model, history, val_ds, w_r2


In [16]:

# ==================== 11B. train CNN Baseline (gram) ====================
set_label_mode(False)  # use gram columns as training target
include_meta: bool = False # <--- 1. tabular switch, image only for baseline

# run 5-fold CV
baseline_fold_maes_gram = []
baseline_models_gram = []
baseline_val_ds_gram = []
baseline_w_r2_gram = []

# give each fold its own random seed
set_seed(CFG.seed)

for f in range(CFG.n_folds):  # CFG.n_folds = 5
# for f in [1]:               # [x] = specific fold number you want to run 
    mae, m, history, val_ds, w_r2 = train_one_fold_baseline(f, save_ckpt=True) # stop passing target_list, instead default to TRAIN_TARGETS
    baseline_fold_maes_gram.append(mae)
    baseline_models_gram.append(m)
    baseline_val_ds_gram.append(val_ds)
    baseline_w_r2_gram.append(w_r2)

baseline_fold_maes_gram = np.array(baseline_fold_maes_gram)
print("\n===== [CNN Baseline - gram] CV summary =====")
print("val_mae_all per fold:", np.round(baseline_fold_maes_gram, 3))
print("mean ± std =", baseline_fold_maes_gram.mean(), "±", baseline_fold_maes_gram.std())
print("weighted_r2 per fold:", np.round(baseline_w_r2_gram, 3))
print("basline_gram mean R2 =", np.mean(baseline_w_r2_gram))


[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
[init] fallback label_transform=identity

===== Fold 0 =====
[Baseline] use_log1p=False, target_list=['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
Fold 0 | Target: ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
  Train: 285 images | Val: 72 images
Fold 0 | Meta=False | Train:285 Val:72
Epoch 1/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 754ms/step - loss: 301.4460 - mae__dry__clover_g: 7.0915 - mae__dry__dead_g: 10.3539 - mae__dry__green_g: 26.2679 - mae__dry__total_g: 36.4703 - mae_all: 21.5627 - mae_gdm_g: 27.6301 - val_loss: 616.6448 - val_mae__dry__clover_g: 25.3887 - val_mae__dry__dead_g: 22.0247 - val_mae__dry__green_g: 25.6285 - val_mae__dry__total_g: 66.4384 - val_mae_all: 35.4413 - val_mae_gdm_g: 37.7261 - learning_rate: 0.0010
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 749ms/step - loss: 143.5224 - mae__dry__clover_g: 9.0

===== [CNN Baseline - gram] CV summary =====3 target
val_mae_all per fold: [10.121 11.914 10.549 10.01  10.141]
mean ± std = 10.54708251953125 ± 0.7075833822485404
weighted_r2 per fold: [ 0.304 -0.171  0.498  0.39   0.33 ]
basline_gram mean R2 = 0.2702052972636771

===== [CNN Baseline - gram] CV summary =====5 target
val_mae_all per fold: [11.332 14.766 12.259 16.392 12.652]
mean ± std = 13.479974555969239 ± 1.8397128198302635
weighted_r2 per fold: [ 0.426  0.287  0.595 -0.147  0.351]
basline_gram mean R2 = 0.30244747404764405

In [17]:
# only to check model structure summary

baseline_model = biomass_conv_baseline( 
    image_size_h=CFG.image_size_h,
    image_size_w=CFG.image_size_w,
    target_names=RAW_TARGETS,
)
baseline_model.summary()


Model: "BiomassConvBaseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 448, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ImageBackbone (Functional)      │ (None, 256)            │       172,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ biomass (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 206,309 (805.89 KB)

 Trainable params: 205,573 (803.02 KB)

 Non-trainable params: 736 (2.88 KB)

____________________________________________________________________________________________________________________________________

____________________________________________________________________________________________________________________________________

Advanced Multi-Modal Biomass Prediction with Knowledge Distillation
====================================================================

Architecture Hierarchy:
1. Hybrid Teacher (Multi-Modal) - uses ALL information
2. Meta-Aware Hybrid Teacher (Enhanced) - with attention mechanisms  
3. Distilled Student (Image-Only) - deployable model
4. Ensemble Strategy - combining multiple approaches


This module implements:
✓ Hybrid Teacher with cross-modal attention
✓ Knowledge distillation framework
✓ Image-only student for deployment
✓ Progressive training pipeline
✓ Ensemble strategies

$$x = \left\{
\begin{array}{ll}
\text{"image"}: & \text{Image Tensor} \\
\text{"meta\_cont"}: & \text{Continuous Features Tensor} \\
\text{"meta\_cat"}: & \text{Categorical Features Tensor}
\end{array}
\right\}$$

In [18]:
# ==================== 12.1. Hybrid Teacher model (Dict Input-key for Multi-Modal Fusion)====================

from tensorflow.keras import models
from tensorflow.keras import layers, models, optimizers, callbacks
# Hybrid Teacher model:
# image_backbone → 256-d tensor
# meta_cont → Dense(32/64) ×2
# meta_cat → Embedding(4+4/8+8) → concat
# concat → Dense(64/128) → Dropout → Dense(3)

def biomass_hybrid_teacher(
    image_size_h: int = CFG.image_size_h,
    image_size_w: int = CFG.image_size_w,
    target_names: list | None = None,
) -> tf.keras.Model:
    """
    Hybrid Teacher:
      Input x: dict
        - x["image"]:     (H, W, 3) Image tensor.
        - x["meta_cont"]: Continuous metadata features (e.g., NDVI, Height, sine/cosine dates).
        - x["meta_cat"]:  Categorical metadata encoded [State_encoded, Species_encoded].
      Output: len(target_names) core targets (default 3).
    """
    if target_names is None:
        target_names = TRAIN_TARGETS
    n_targets = len(target_names)

    # Dictionary Inputs (Defining the expected inputs for the Keras Functional API)
    inputs = {
        "image":     layers.Input(shape=(image_size_h, image_size_w, 3), name="image"),
        "meta_cont": layers.Input(shape=(len(META_COLS_CONT),),  name="meta_cont"),
        "meta_cat":  layers.Input(shape=(len(META_COLS_CAT),), dtype="int32", name="meta_cat"),
    }

    # 1) image backbone
    # This block uses the predefined CNN structure (assumed to be similar to BiomassConvBaseline).
    backbone = build_image_backbone(image_size_h, image_size_w, name="ImageBackboneTeacher")
    img_feat = backbone(inputs["image"])   # Output: (Batch, 256) feature vector

    # 2) Continuous Features MLP Branch
    # Processes numerical (continuous) metadata features like NDVI and Height.
    cont_x = layers.Dense(64, activation="relu")(inputs["meta_cont"])  # 32->64
    cont_x = layers.BatchNormalization()(cont_x) 
    cont_x = layers.Dense(64, activation="relu")(cont_x)  # 32->64
    cont_x = layers.Dropout(0.2)(cont_x) 

    # 3) Categorical Features Embedding Branch
    # Processes encoded categorical features (State and Species).
    cat_inputs = inputs["meta_cat"]  # Input tensor shape is (batch, 2) 
    
    # Slicing the tensor to feed individual columns into separate embedding layers
    # defaultlt META_COLS_CAT = ["State_encoded", "Species_encoded"] 
    state_ids   = layers.Lambda(lambda x: x[:, 0])(cat_inputs) # State_encoded->outcome 0, 1, 2, ..., n-1 ontinuous integers
    species_ids = layers.Lambda(lambda x: x[:, 1])(cat_inputs) # Species_encoded
    
    # Define embedding dimensions
    state_emb_dim   = 8 
    species_emb_dim = 8 
    
    # Create and apply Embedding layers (converts integer codes to dense vectors)
    state_emb_layer   = layers.Embedding(len(STATE_MAP),   state_emb_dim,   name="state_emb") 
    species_emb_layer = layers.Embedding(len(SPECIES_MAP), species_emb_dim, name="species_emb")

    state_emb   = state_emb_layer(state_ids)     # Output:(batch, 8)
    species_emb = species_emb_layer(species_ids) # Output:(batch, 8)
    
    # Combine all categorical embeddings
    cat_emb = layers.Concatenate(name="cat_emb")([state_emb, species_emb])  # Output: (batch, 16)
    
    # Combine Continuous MLP output and Categorical Embeddings to form Tabular Feature Vector
    tab_x = layers.Concatenate(name="tabular_fused")([cont_x, cat_emb])

    # 4) Image + Tabular Fusion & Regression Head
    # Concatenate the high-level image features with the processed tabular features.
    fused = layers.Concatenate(name="img_tab_fusion")([img_feat, tab_x])
    
    # Final regression head MLP
    x = layers.Dense(128, activation="relu")(fused)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(
        n_targets,
        activation="softplus", # Softplus ensures non-negative biomass predictions - softplus(z) = log(1 + e^z)
        name="biomass",
        dtype="float32",
    )(x)
    
    # Construct the model
    model = models.Model(inputs=inputs, outputs=outputs, name="HybridTeacher")
    
    # 5) Compile Model
    # Reuse existing weighted loss function and MAE metrics.
    loss_weights = get_loss_weights(target_names)

    model.compile(
        optimizer=optimizers.Adam(1e-3),
        loss=weighted_mse_loss(loss_weights, max_err=200.0),
        metrics=make_mae_metrics(target_names),
    )
    return model


In [19]:

def train_one_fold_teacher(
    fold,
    target_list=None,
    model_fn=biomass_hybrid_teacher,
    **model_kwargs,
):
    """
    Hybrid Teacher: image + metadata model.
    """
    return train_one_fold_generic(
        fold=fold,
        model_fn=model_fn,
        include_meta=True,      # image + tabular
        target_list=target_list,
        model_name="Teacher",
        **model_kwargs,
    )

In [20]:

# ====== run model: Hybrid Teacher (gram)======
set_label_mode(False)  # Set label mode to "gram"
# set_label_mode(True)  # Set label mode to "log1p"

teacher_fold_maes_gram = []
teacher_models_gram = []
teacher_val_ds_gram = []
teacher_w_r2_gram = []

set_seed(CFG.seed)

for f in range(CFG.n_folds):
# for f in [1]:
    mae, m, history, val_ds, w_r2 = train_one_fold_teacher(f)
    teacher_fold_maes_gram.append(mae)
    teacher_models_gram.append(m)
    teacher_val_ds_gram.append(val_ds)
    teacher_w_r2_gram.append(w_r2)

teacher_fold_maes_gram = np.array(teacher_fold_maes_gram)
print("\n===== [Teacher] 5-fold CV summary =====")
print("val_mae_all per fold:", np.round(teacher_fold_maes_gram, 3))
print("mean ± std =", teacher_fold_maes_gram.mean(), "±", teacher_fold_maes_gram.std())
print("weighted_r2 per fold:", np.round(teacher_w_r2_gram, 3))
print("teacher mean R2 =", np.mean(teacher_w_r2_gram))


[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
[init] fallback label_transform=identity

===== Fold 0 =====
[Baseline] use_log1p=False, target_list=['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
Fold 0 | Target: ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
  Train: 285 images | Val: 72 images
Fold 0 | Meta=True | Train:285 Val:72
Epoch 1/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 753ms/step - loss: 267.3400 - mae__dry__clover_g: 6.7737 - mae__dry__dead_g: 11.3246 - mae__dry__green_g: 23.0752 - mae__dry__total_g: 31.7152 - mae_all: 20.8870 - mae_gdm_g: 31.5465 - val_loss: 456.4540 - val_mae__dry__clover_g: 12.1457 - val_mae__dry__dead_g: 12.7933 - val_mae__dry__green_g: 24.8311 - val_mae__dry__total_g: 59.9353 - val_mae_all: 25.0651 - val_mae_gdm_g: 15.6202 - learning_rate: 0.0010
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 747ms/step - loss: 102.0738 - mae__dry__clover_g: 8.61

===== [Teacher] 5-fold CV summary =====
val_mae_all per fold: [7.32  6.695 6.755 7.598 7.319]
mean ± std = 7.137342548370361 ± 0.35214325238677197
weighted_r2 per fold: [0.531 0.741 0.824 0.672 0.728]
teacher mean R2 = 0.6991199103874223

===== [Teacher] 5-fold CV summary =====
val_mae_all per fold: [8.679 8.835 7.733 8.014 8.233]
mean ± std = 8.299013805389404 ± 0.40934585632045123
weighted_r2 per fold: [0.598 0.702 0.855 0.742 0.802]
teacher mean R2 = 0.7396731941512208

In [21]:
# only to check model structure summary

teacher_model = biomass_hybrid_teacher( 
    image_size_h=CFG.image_size_h,
    image_size_w=CFG.image_size_w,
    target_names=RAW_TARGETS,
)
teacher_model.summary()

Model: "HybridTeacher"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ meta_cont           │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 64)        │        512 │ meta_cont[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meta_cat            │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_21[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_10 (Lambda)  │ (None)            │          0 │ meta_cat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_11 (Lambda)  │ (None)            │          0 │ meta_cat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 64)        │      4,160 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ state_emb           │ (None, 8)         │         32 │ lambda_10[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ species_emb         │ (None, 8)         │        120 │ lambda_11[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image (InputLayer)  │ (None, 224, 448,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 64)        │          0 │ dense_22[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_emb             │ (None, 16)        │          0 │ state_emb[0][0],  │
│ (Concatenate)       │                   │            │ species_emb[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ImageBackboneTeach… │ (None, 256)       │    172,768 │ image[0][0]       │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tabular_fused       │ (None, 80)        │          0 │ dropout_16[0][0], │
│ (Concatenate)       │                   │            │ cat_emb[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_tab_fusion      │ (None, 336)       │          0 │ ImageBackboneTea… │
│ (Concatenate)       │                   │            │ tabular_fused[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 128)       │     43,136 │ img_tab_fusion[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 128)       │          0 │ dense_23[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ biomass (Dense)     │ (None, 5)         │        645 │ dropout_17[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 221,629 (865.74 KB)

 Trainable params: 220,765 (862.36 KB)

 Non-trainable params: 864 (3.38 KB)

____________________________________________________________________________________________________________________________________

In [54]:

# ==================== 12.2. Student model ====================
# Distillation Loss for Student(image-only)
def distillation_loss(
    target_names,
    alpha=0.4,      # weight for teacher loss  alpha ∈ {0.2, 0.3, 0.5}  
    tau=1.0,        # temperature for distillation
):
    """Create a distillation loss combining GT and teacher predictions.

    Args:
        alpha: Weight of the teacher term in the final loss
            (0 → pure GT, 1 → pure teacher).
        tau: Temperature used to soften student/teacher outputs.
        n_targets: Number of target dimensions.

    Returns:
        Callable: Loss function `loss_fn(y_true_concat, y_pred)` where
        `y_true_concat = [y_gt, y_teacher]` along the last dimension.
        
    Note:
    mse_distill ≈ (1 / tau**2) * MSE(y_pred, y_teacher)
    So the effective weights are:
    Loss ≈ (1 - alpha) * L_gt + alpha / (tau**2) * L_teacher
    """
    
    n_targets = len(target_names)
    loss_weights = tf.constant(get_loss_weights(target_names), tf.float32)

    def loss_fn(y_true_concat, y_pred):
        # Split ground truth and teacher predictions
        y_true    = y_true_concat[:, :n_targets]
        y_teacher = y_true_concat[:, n_targets:]
        
        # Student vs GT weighted MSE
        err_gt = tf.square(y_pred - y_true)
        mse_gt = tf.reduce_mean(err_gt * loss_weights, axis=-1) 
        
        # Student vs teacher weighted MSE (with temperature)
        y_pred_t    = y_pred    / tau
        y_teacher_t = y_teacher / tau
        err_distill = tf.square(y_pred_t - y_teacher_t)
        mse_distill = tf.reduce_mean(err_distill * loss_weights, axis=-1) 
        
        # Total loss = (1 - alpha) * GT + alpha * teacher
        return (1.0 - alpha) * mse_gt + alpha * mse_distill

    return loss_fn



In [23]:
def biomass_student_from_baseline(
    image_size_h: int = CFG.image_size_h, 
    image_size_w: int = CFG.image_size_w, 
    target_names: list | None = None,
    baseline_weights_path: str | None = None,
    distill_alpha: float = 0.4,
     distill_tau: float = 1.0, 
    use_distill_loss: bool = True,
) -> tf.keras.Model:
    """Build the student CNN model initialized from the baseline backbone.

    The student is an image-only CNN. Optionally, it:
      * Loads backbone weights from a pre-trained baseline model.
      * Uses a distillation loss that mixes GT and teacher predictions.

    Args:
        image_size_h: Input image height in pixels.
        image_size_w: Input image width in pixels.
        target_names: List of target column names; if None, defaults to TRAIN_TARGETS.
        baseline_weights_path: Optional path to baseline model weights (.h5). If
            provided, the student backbone is initialized from these weights.
        distill_alpha: Weight of the teacher term in the distillation loss.
        distill_tau: Temperature for the distillation loss.
        use_distill_loss: If True, use distillation loss; otherwise, use
            weighted MSE on ground-truth only.

    Returns:
        tf.keras.Model: Compiled student model ready for training.
    """
    if target_names is None:
        target_names = TRAIN_TARGETS
    n_targets = len(target_names)

    # 1) Student backbone (image-only)
    backbone = build_image_backbone(image_size_h, image_size_w, name="ImageBackboneStudent")
    img_inputs = layers.Input(shape=(image_size_h, image_size_w, 3), name="image")
    x = backbone(img_inputs)

    # 2) Regression head (can mirror baseline or be slightly deeper)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    # Optionally add another layer:
    # x = layers.Dense(64, activation="relu")(x)
    # x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(
        n_targets,
        activation="softplus",
        name="biomass",
        dtype="float32",
    )(x)

    model = models.Model(
        inputs=img_inputs, outputs=outputs, name="StudentFromBaseline"
    )

    # 3) If baseline weights are provided, copy its backbone weights
    if baseline_weights_path is not None:
        print(f"[Student] load baseline backbone from {baseline_weights_path}")
        # Create a temporary baseline model to load weights into
        baseline_tmp = biomass_conv_baseline(
            image_size_h=image_size_h,
            image_size_w=image_size_w,
            target_names=target_names,
        )
        baseline_tmp.load_weights(baseline_weights_path)

        # Extract backbones
        baseline_backbone = baseline_tmp.get_layer("ImageBackbone")
        student_backbone  = model.get_layer("ImageBackboneStudent")

         # Copy CNN backbone parameters
        student_backbone.set_weights(baseline_backbone.get_weights())
        print("[Student] backbone weights copied from baseline")

        # Free the temporary model
        del baseline_tmp

     # 4) Choose loss: distillation vs pure GT (weighted MSE)
    loss_weights = get_loss_weights(target_names)

    if use_distill_loss:
        # this alpha = “teacher w”
        loss_fn = distillation_loss(
            alpha=distill_alpha,
            tau=distill_tau,  
            target_names=target_names,
        )
    else:
        loss_fn = weighted_mse_loss(loss_weights, max_err=200.0)

    model.compile(
        optimizer=optimizers.Adam(1e-4),   # Finetuning LR is smaller than baseline
        loss=loss_fn,
        metrics=make_mae_metrics(target_names),
    )
    return model


In [ ]:
def run_one_fold_student_from_baseline_no_distill(fold):
    """Train a student initialized from baseline backbone without distillation.

    The student backbone is initialized from the baseline checkpoint and then
    trained using only ground-truth supervision (no distillation loss).

    Args:
        fold: Fold index used as validation split.

    Returns:
        tuple: (best_val_mae, model, history, val_ds, w_r2)
    """
    baseline_ckpt = f"./ckpt/baseline_gram_fold{fold}.weights.h5"
    
    best_val_mae, model, history, val_ds, w_r2 = train_one_fold_generic(
        fold=fold,
        include_meta=False,   # student is also image-only
        model_fn=biomass_student_from_baseline,
        model_name=f"student_from_baseline_fold{fold}",
        baseline_weights_path=baseline_ckpt,
        distill_alpha=0.0,
        use_distill_loss=False,   # no distillation/teacher loss here
    )
    return best_val_mae, model, history, val_ds, w_r2


In [ ]:
# ====== Train model: no-Distilled Student (image only)(gram) ======
set_label_mode(False)  # Set label mode to "gram"
# set_label_mode(True)  # Set label mode to "log1p"

student_results_gram_no_distill = {}
student_fold_maes_gram_no_distill = []
student_fold_r2_gram_no_distill = []

set_seed(CFG.seed)

for f in range(CFG.n_folds):
    
    student_mae, student_model, student_hist, student_valds, student_r2 = \
        run_one_fold_student_from_baseline_no_distill(
            fold=f,
            # teacher_model=teacher_models_gram[f],   # trained teacher
            # alpha=0.0,
            # tau=5.0,
        )
    student_results_gram_no_distill[f] = (student_mae, student_r2)
    student_fold_maes_gram_no_distill.append(student_mae)
    student_fold_r2_gram_no_distill.append(student_r2)

print("\n===== [Student Distill] 5-fold CV summary =====")
print("MAE per fold:", np.round(student_fold_maes_gram_no_distill, 3))
print("R2  per fold:", np.round(student_fold_r2_gram_no_distill, 3))
print("mean R2 =", np.mean(student_fold_r2_gram_no_distill))

[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
[init] fallback label_transform=identity

===== Fold 0 =====
[Baseline] use_log1p=False, target_list=['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
Fold 0 | Target: ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
  Train: 285 images | Val: 72 images
Fold 0 | Meta=False | Train:285 Val:72
[Student] load baseline backbone from ./ckpt/baseline_gram_fold0.weights.h5
[Student] backbone weights copied from baseline
Epoch 1/30


/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 782ms/step - loss: 372.2223 - mae__dry__clover_g: 6.7088 - mae__dry__dead_g: 11.2861 - mae__dry__green_g: 26.4120 - mae__dry__total_g: 43.8675 - mae_all: 23.9814 - mae_gdm_g: 31.6327 - val_loss: 352.9891 - val_mae__dry__clover_g: 5.6177 - val_mae__dry__dead_g: 13.1163 - val_mae__dry__green_g: 25.2714 - val_mae__dry__total_g: 43.5591 - val_mae_all: 23.4189 - val_mae_gdm_g: 29.5299 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 777ms/step - loss: 348.5136 - mae__dry__clover_g: 6.6447 - mae__dry__dead_g: 11.1275 - mae__dry__green_g: 26.4756 - mae__dry__total_g: 41.8342 - mae_all: 23.4083 - mae_gdm_g: 30.9596 - val_loss: 338.1452 - val_mae__dry__clover_g: 5.6046 - val_mae__dry__dead_g: 12.9674 - val_mae__dry__green_g: 25.2811 - val_mae__dry__total_g: 42.1340 - val_mae_all: 23.0736 - val_mae_gdm_g: 29.3808 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 778ms/step - loss: 321.1686 - mae__dry__clover_g: 6.7450 - mae_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 746ms/step - loss: 384.3039 - mae__dry__clover_g: 6.3734 - mae__dry__dead_g: 11.4192 - mae__dry__green_g: 26.4009 - mae__dry__total_g: 44.4904 - mae_all: 24.1793 - mae_gdm_g: 32.2128 - val_loss: 341.3756 - val_mae__dry__clover_g: 7.0475 - val_mae__dry__dead_g: 10.3925 - val_mae__dry__green_g: 25.3311 - val_mae__dry__total_g: 42.9774 - val_mae_all: 23.6232 - val_mae_gdm_g: 32.3675 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 27s 743ms/step - loss: 363.3276 - mae__dry__clover_g: 6.3711 - mae__dry__dead_g: 11.0696 - mae__dry__green_g: 26.4303 - mae__dry__total_g: 42.4329 - mae_all: 23.6194 - mae_gdm_g: 31.7934 - val_loss: 320.2050 - val_mae__dry__clover_g: 7.0455 - val_mae__dry__dead_g: 9.9616 - val_mae__dry__green_g: 25.2532 - val_mae__dry__total_g: 40.6538 - val_mae_all: 23.0039 - val_mae_gdm_g: 32.1057 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 27s 742ms/step - loss: 333.9388 - mae__dry__clover_g: 6.3589 - mae__

...

36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 750ms/step - loss: 354.0236 - mae__dry__clover_g: 6.2498 - mae__dry__dead_g: 11.9946 - mae__dry__green_g: 24.8917 - mae__dry__total_g: 43.1845 - mae_all: 23.5667 - mae_gdm_g: 31.5128 - val_loss: 401.6042 - val_mae__dry__clover_g: 7.5012 - val_mae__dry__dead_g: 9.8379 - val_mae__dry__green_g: 27.8776 - val_mae__dry__total_g: 43.6507 - val_mae_all: 24.8734 - val_mae_gdm_g: 35.4994 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 744ms/step - loss: 324.9503 - mae__dry__clover_g: 6.2158 - mae__dry__dead_g: 11.8538 - mae__dry__green_g: 24.3259 - mae__dry__total_g: 40.5103 - mae_all: 22.7790 - mae_gdm_g: 30.9889 - val_loss: 353.9995 - val_mae__dry__clover_g: 7.4583 - val_mae__dry__dead_g: 9.6724 - val_mae__dry__green_g: 26.8993 - val_mae__dry__total_g: 39.5491 - val_mae_all: 23.6198 - val_mae_gdm_g: 34.5201 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 290.2296 - mae__dry__clover_g: 6.2140 - mae__d

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 31s 789ms/step - loss: 375.4732 - mae__dry__clover_g: 6.3353 - mae__dry__dead_g: 11.3504 - mae__dry__green_g: 26.0848 - mae__dry__total_g: 43.4402 - mae_all: 23.9997 - mae_gdm_g: 32.7876 - val_loss: 321.2035 - val_mae__dry__clover_g: 6.9768 - val_mae__dry__dead_g: 12.8289 - val_mae__dry__green_g: 22.7503 - val_mae__dry__total_g: 41.1746 - val_mae_all: 23.0284 - val_mae_gdm_g: 31.4114 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 782ms/step - loss: 348.4005 - mae__dry__clover_g: 6.2799 - mae__dry__dead_g: 11.4224 - mae__dry__green_g: 25.5672 - mae__dry__total_g: 40.5837 - mae_all: 23.2913 - mae_gdm_g: 32.6035 - val_loss: 269.6367 - val_mae__dry__clover_g: 6.8900 - val_mae__dry__dead_g: 13.0172 - val_mae__dry__green_g: 21.1181 - val_mae__dry__total_g: 35.0096 - val_mae_all: 21.4967 - val_mae_gdm_g: 31.4485 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 782ms/step - loss: 319.1549 - mae__dry__clover_g: 6.3402 - mae_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 776ms/step - loss: 360.6901 - mae__dry__clover_g: 6.7597 - mae__dry__dead_g: 11.3325 - mae__dry__green_g: 25.6998 - mae__dry__total_g: 43.4563 - mae_all: 23.9079 - mae_gdm_g: 32.2912 - val_loss: 370.9169 - val_mae__dry__clover_g: 5.4097 - val_mae__dry__dead_g: 10.6636 - val_mae__dry__green_g: 27.5246 - val_mae__dry__total_g: 41.3200 - val_mae_all: 23.5099 - val_mae_gdm_g: 32.6316 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 776ms/step - loss: 328.8571 - mae__dry__clover_g: 6.7979 - mae__dry__dead_g: 11.0132 - mae__dry__green_g: 25.6273 - mae__dry__total_g: 40.4203 - mae_all: 23.1579 - mae_gdm_g: 31.9310 - val_loss: 325.7004 - val_mae__dry__clover_g: 5.3914 - val_mae__dry__dead_g: 10.1952 - val_mae__dry__green_g: 27.4897 - val_mae__dry__total_g: 36.5984 - val_mae_all: 22.3297 - val_mae_gdm_g: 31.9739 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 775ms/step - loss: 295.1169 - mae__dry__clover_g: 6.7976 - mae_

===== [Student no-Distill] 5-fold CV summary ===== 3 target
MAE per fold: [ 9.151 10.531 10.454  9.288  9.149]
R2  per fold: [0.383 0.414 0.353 0.48  0.231]
mean R2 = 0.37238640442242643

===== [Student no-Distill] 5-fold CV summary ===== 5 target
MAE per fold: [11.306 12.196 12.005 12.301 12.586]
R2  per fold: [0.469 0.419 0.64  0.395 0.362]
mean R2 = 0.4569973935200549

--------------------------------------

In [57]:
def train_one_fold_student_distill(
    fold,
    teacher_model,
    alpha=0.4,
    tau=3.0,
    target_list=None,
    model_fn=biomass_student_from_baseline,
    **model_kwargs,
):
    """Train a distilled student for one fold using a fixed teacher model.

    Args:
        fold: Fold index used as validation split.
        teacher_model: Trained teacher model (image + metadata) used to
            generate soft labels.
        alpha: Distillation weight (teacher term) passed to the loss.
        tau: Distillation temperature passed to the loss.
        target_list: List of target column names; if None, defaults to TRAIN_TARGETS.
        model_fn: Function that builds the student model.
        **model_kwargs: Extra keyword arguments forwarded to `model_fn`.

    Returns:
        tuple: (best_val_mae, student_model, history, val_ds, w_r2)
    """
    
    if target_list is None:
        target_list = TRAIN_TARGETS

    print(f"\n===== [Student Distill] Fold {fold} (alpha={alpha}, tau={tau}) =====")

    # 1) Build data module and collect all data
    dm = BiomassImgDataModule(fold=fold, target_list=target_list, include_meta=True)
    imgs_all = dm.make_arrays()          # (N, H, W, 3)
    y_gt_all = dm.y                     # (N, T)

    meta_cont_all = dm.df[dm.meta_cols_cont].astype("float32").values
    meta_cat_all  = dm.df[dm.meta_cols_cat ].astype("int32").values

    # 2) Teacher predictions for all samples 
    x_all = {
        "image":     imgs_all,
        "meta_cont": meta_cont_all,
        "meta_cat":  meta_cat_all,
    }
    teacher_preds_all = teacher_model.predict(
        x_all,
        batch_size=CFG.batch_size,
        verbose=1,   # (N, T)
    )                                 

    # 3) Concatenate GT and teacher predictions: y_true_concat = [y_gt, y_teacher]
    y_concat_all = np.concatenate([y_gt_all, teacher_preds_all], axis=1)  # (N, 2T)

    # 4) Train / Val split
    val_mask = (dm.df["fold"] == fold).values
    tr_idx = np.where(~val_mask)[0]
    va_idx = np.where(val_mask)[0]

    print(f"[Student] Fold {fold} Train: {len(tr_idx)} | Val: {len(va_idx)}")

    train_ds = tf.data.Dataset.from_tensor_slices(
        (imgs_all[tr_idx], y_concat_all[tr_idx])
    )
    val_ds = tf.data.Dataset.from_tensor_slices(
        (imgs_all[va_idx], y_concat_all[va_idx])
    )

    train_ds = (
        train_ds
        .map(tfdata_augmentor, num_parallel_calls=1)
        .shuffle(1024, seed=CFG.seed, reshuffle_each_iteration=False)
        .batch(CFG.batch_size)
        .prefetch(1)
    )
    val_ds = (
        val_ds
        .map(tfdata_clip, num_parallel_calls=1)
        .batch(CFG.batch_size)
        .prefetch(1)
    )

    # 5) Build student-from-baseline + distillation loss
    mode = "log1p" if CFG.use_log1p else "gram"
    baseline_ckpt = f"./ckpt/baseline_{mode}_fold{fold}.weights.h5"

    student_model = model_fn(
        image_size_h=CFG.image_size_h,
        image_size_w=CFG.image_size_w,
        target_names=target_list,
        baseline_weights_path=baseline_ckpt,
        distill_alpha=alpha, # teacher weight in distillation loss
        distill_tau=tau,      # biomass_student_from_baseline 
        use_distill_loss=True,
        **model_kwargs,
    )

    # 6) callbacks
    callbacks_list = build_callbacks(monitor="val_mae_all", mode="min")

    # 7) train
    history = student_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        callbacks=callbacks_list,
        verbose=1,
    )

    best_val_mae = min(history.history["val_mae_all"])
    print(f"[Student] Fold {fold} best val_mae_all = {best_val_mae:.3f}")

    # 8) Evaluate in gram space
    val_df, x_val = build_val_inputs(
        dm=dm,
        fold=fold,
        include_meta=False,   # student uses image only for inference
    )
    y_pred_default = student_model.predict(x_val, batch_size=CFG.batch_size)

    w_r2, pred_df, true_df = eval_weighted_r2_in_gram(
        val_df=val_df,
        y_pred_default=y_pred_default,
        model_name="Student",
        fold=fold,
        # target_list=target_list,
    )

    return best_val_mae, student_model, history, val_ds, w_r2


In [30]:
# ====== Train model: Distilled Student (image only)(gram) ======
set_label_mode(False)  # Set label mode to "gram"
# set_label_mode(True)  # Set label mode to "log1p"

student_results_gram = {}
student_fold_maes_gram = []
student_fold_r2_gram = []

set_seed(CFG.seed)

for f in range(CFG.n_folds):
    
    student_mae, student_model, student_hist, student_valds, student_r2 = \
        train_one_fold_student_distill(
            fold=f,
            teacher_model=teacher_models_gram[f],   # trained teacher fisrt
            alpha=0.5,
            tau=1.0,
        )
    student_results_gram[f] = (student_mae, student_r2)
    student_fold_maes_gram.append(student_mae)
    student_fold_r2_gram.append(student_r2)

print("\n===== [Student Distill] 5-fold CV summary =====")
print("MAE per fold:", np.round(student_fold_maes_gram, 3))
print("R2  per fold:", np.round(student_fold_r2_gram, 3))
print("mean R2 =", np.mean(student_fold_r2_gram))


[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
[init] fallback label_transform=identity

===== [Student Distill] Fold 0 (alpha=0.5, tau=1.0) =====
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 200ms/step
[Student] Fold 0 Train: 285 | Val: 72
[Student] load baseline backbone from ./ckpt/baseline_gram_fold0.weights.h5
[Student] backbone weights copied from baseline
Epoch 1/30


/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 765ms/step - loss: 358.1673 - mae__dry__clover_g: 6.7097 - mae__dry__dead_g: 11.2876 - mae__dry__green_g: 26.4126 - mae__dry__total_g: 43.8530 - mae_all: 23.9791 - mae_gdm_g: 31.6325 - val_loss: 335.0154 - val_mae__dry__clover_g: 5.6195 - val_mae__dry__dead_g: 13.1183 - val_mae__dry__green_g: 25.2691 - val_mae__dry__total_g: 43.5578 - val_mae_all: 23.4184 - val_mae_gdm_g: 29.5274 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 761ms/step - loss: 334.3819 - mae__dry__clover_g: 6.6465 - mae__dry__dead_g: 11.1237 - mae__dry__green_g: 26.4759 - mae__dry__total_g: 41.7957 - mae_all: 23.4004 - mae_gdm_g: 30.9601 - val_loss: 320.1736 - val_mae__dry__clover_g: 5.6058 - val_mae__dry__dead_g: 12.9639 - val_mae__dry__green_g: 25.2763 - val_mae__dry__total_g: 42.1608 - val_mae_all: 23.0779 - val_mae_gdm_g: 29.3827 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 757ms/step - loss: 306.8572 - mae__dry__clover_g: 6.7531 - mae_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 764ms/step - loss: 360.5936 - mae__dry__clover_g: 6.3729 - mae__dry__dead_g: 11.4139 - mae__dry__green_g: 26.4001 - mae__dry__total_g: 44.4755 - mae_all: 24.1747 - mae_gdm_g: 32.2113 - val_loss: 331.4297 - val_mae__dry__clover_g: 7.0454 - val_mae__dry__dead_g: 10.3648 - val_mae__dry__green_g: 25.3204 - val_mae__dry__total_g: 42.9328 - val_mae_all: 23.6080 - val_mae_gdm_g: 32.3769 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 752ms/step - loss: 339.2737 - mae__dry__clover_g: 6.3686 - mae__dry__dead_g: 11.0435 - mae__dry__green_g: 26.4299 - mae__dry__total_g: 42.3628 - mae_all: 23.6017 - mae_gdm_g: 31.8038 - val_loss: 309.7262 - val_mae__dry__clover_g: 7.0490 - val_mae__dry__dead_g: 9.8922 - val_mae__dry__green_g: 25.2560 - val_mae__dry__total_g: 40.5313 - val_mae_all: 22.9724 - val_mae_gdm_g: 32.1336 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 755ms/step - loss: 309.6623 - mae__dry__clover_g: 6.3601 - mae__

...

36/36 ━━━━━━━━━━━━━━━━━━━━ 31s 802ms/step - loss: 333.5544 - mae__dry__clover_g: 6.2506 - mae__dry__dead_g: 11.9917 - mae__dry__green_g: 24.8854 - mae__dry__total_g: 43.1721 - mae_all: 23.5632 - mae_gdm_g: 31.5163 - val_loss: 369.1415 - val_mae__dry__clover_g: 7.4941 - val_mae__dry__dead_g: 9.8263 - val_mae__dry__green_g: 27.8348 - val_mae__dry__total_g: 43.5034 - val_mae_all: 24.8300 - val_mae_gdm_g: 35.4912 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 790ms/step - loss: 304.0817 - mae__dry__clover_g: 6.2171 - mae__dry__dead_g: 11.8372 - mae__dry__green_g: 24.3086 - mae__dry__total_g: 40.4529 - mae_all: 22.7611 - mae_gdm_g: 30.9896 - val_loss: 319.5610 - val_mae__dry__clover_g: 7.4404 - val_mae__dry__dead_g: 9.6271 - val_mae__dry__green_g: 26.7611 - val_mae__dry__total_g: 39.1147 - val_mae_all: 23.4804 - val_mae_gdm_g: 34.4584 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 785ms/step - loss: 268.8802 - mae__dry__clover_g: 6.2177 - mae__d

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 754ms/step - loss: 346.3378 - mae__dry__clover_g: 6.3350 - mae__dry__dead_g: 11.3510 - mae__dry__green_g: 26.0833 - mae__dry__total_g: 43.4274 - mae_all: 23.9974 - mae_gdm_g: 32.7902 - val_loss: 309.4688 - val_mae__dry__clover_g: 6.9703 - val_mae__dry__dead_g: 12.8316 - val_mae__dry__green_g: 22.7058 - val_mae__dry__total_g: 41.0474 - val_mae_all: 22.9939 - val_mae_gdm_g: 31.4142 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 746ms/step - loss: 320.0337 - mae__dry__clover_g: 6.2784 - mae__dry__dead_g: 11.4233 - mae__dry__green_g: 25.5560 - mae__dry__total_g: 40.5410 - mae_all: 23.2810 - mae_gdm_g: 32.6064 - val_loss: 256.2596 - val_mae__dry__clover_g: 6.9047 - val_mae__dry__dead_g: 13.0206 - val_mae__dry__green_g: 20.9824 - val_mae__dry__total_g: 34.5127 - val_mae_all: 21.3688 - val_mae_gdm_g: 31.4239 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 755ms/step - loss: 291.5620 - mae__dry__clover_g: 6.3435 - mae_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 785ms/step - loss: 345.3598 - mae__dry__clover_g: 6.7598 - mae__dry__dead_g: 11.3299 - mae__dry__green_g: 25.6999 - mae__dry__total_g: 43.4435 - mae_all: 23.9050 - mae_gdm_g: 32.2920 - val_loss: 333.5408 - val_mae__dry__clover_g: 5.4132 - val_mae__dry__dead_g: 10.6489 - val_mae__dry__green_g: 27.5334 - val_mae__dry__total_g: 41.2042 - val_mae_all: 23.4821 - val_mae_gdm_g: 32.6106 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 782ms/step - loss: 313.2156 - mae__dry__clover_g: 6.8003 - mae__dry__dead_g: 10.9958 - mae__dry__green_g: 25.6296 - mae__dry__total_g: 40.3630 - mae_all: 23.1434 - mae_gdm_g: 31.9282 - val_loss: 281.4457 - val_mae__dry__clover_g: 5.4021 - val_mae__dry__dead_g: 10.0205 - val_mae__dry__green_g: 27.4818 - val_mae__dry__total_g: 35.6633 - val_mae_all: 22.0799 - val_mae_gdm_g: 31.8320 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 781ms/step - loss: 279.3792 - mae__dry__clover_g: 6.8003 - mae_

In [60]:
# ====== Train model: Distilled Student (image only)(gram) ======
set_label_mode(False)  # Set label mode to "gram"
# set_label_mode(True)  # Set label mode to "log1p"

student_results_gram_2 = {}
student_fold_maes_gram_2 = []
student_fold_r2_gram_2 = []

set_seed(CFG.seed)

for f in range(CFG.n_folds):
    
    student_mae, student_model, student_hist, student_valds, student_r2 = \
        train_one_fold_student_distill(
            fold=f,
            teacher_model=teacher_models_gram[f],   # trained teacher fisrt
            alpha=0.55,
            tau=1.0,
        )
    student_results_gram_2[f] = (student_mae, student_r2)
    student_fold_maes_gram_2.append(student_mae)
    student_fold_r2_gram_2.append(student_r2)

print("\n===== [Student Distill] 5-fold CV summary =====")
print("MAE per fold:", np.round(student_fold_maes_gram_2, 3))
print("R2  per fold:", np.round(student_fold_r2_gram_2, 3))
print("mean R2 =", np.mean(student_fold_r2_gram_2))


[set_label_mode] use_log1p=False → TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
[init] fallback label_transform=identity

===== [Student Distill] Fold 0 (alpha=0.55, tau=1.0) =====
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 197ms/step
[Student] Fold 0 Train: 285 | Val: 72
[Student] load baseline backbone from ./ckpt/baseline_gram_fold0.weights.h5
[Student] backbone weights copied from baseline
Epoch 1/30


/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 756ms/step - loss: 356.7682 - mae__dry__clover_g: 6.7097 - mae__dry__dead_g: 11.2878 - mae__dry__green_g: 26.4126 - mae__dry__total_g: 43.8522 - mae_all: 23.9789 - mae_gdm_g: 31.6322 - val_loss: 333.1559 - val_mae__dry__clover_g: 5.6201 - val_mae__dry__dead_g: 13.1199 - val_mae__dry__green_g: 25.2704 - val_mae__dry__total_g: 43.5535 - val_mae_all: 23.4165 - val_mae_gdm_g: 29.5188 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 756ms/step - loss: 332.9925 - mae__dry__clover_g: 6.6469 - mae__dry__dead_g: 11.1239 - mae__dry__green_g: 26.4760 - mae__dry__total_g: 41.7947 - mae_all: 23.4003 - mae_gdm_g: 30.9600 - val_loss: 318.1068 - val_mae__dry__clover_g: 5.6063 - val_mae__dry__dead_g: 12.9644 - val_mae__dry__green_g: 25.2768 - val_mae__dry__total_g: 42.1398 - val_mae_all: 23.0714 - val_mae_gdm_g: 29.3699 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 753ms/step - loss: 305.4496 - mae__dry__clover_g: 6.7542 - mae_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 739ms/step - loss: 358.2258 - mae__dry__clover_g: 6.3727 - mae__dry__dead_g: 11.4142 - mae__dry__green_g: 26.4005 - mae__dry__total_g: 44.4741 - mae_all: 24.1746 - mae_gdm_g: 32.2116 - val_loss: 330.4433 - val_mae__dry__clover_g: 7.0451 - val_mae__dry__dead_g: 10.3653 - val_mae__dry__green_g: 25.3214 - val_mae__dry__total_g: 42.9286 - val_mae_all: 23.6078 - val_mae_gdm_g: 32.3783 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 27s 735ms/step - loss: 336.8837 - mae__dry__clover_g: 6.3684 - mae__dry__dead_g: 11.0428 - mae__dry__green_g: 26.4307 - mae__dry__total_g: 42.3570 - mae_all: 23.6010 - mae_gdm_g: 31.8059 - val_loss: 308.7046 - val_mae__dry__clover_g: 7.0498 - val_mae__dry__dead_g: 9.8885 - val_mae__dry__green_g: 25.2581 - val_mae__dry__total_g: 40.5209 - val_mae_all: 22.9708 - val_mae_gdm_g: 32.1366 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 27s 738ms/step - loss: 307.2593 - mae__dry__clover_g: 6.3603 - mae__

...

36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 770ms/step - loss: 331.5127 - mae__dry__clover_g: 6.2506 - mae__dry__dead_g: 11.9915 - mae__dry__green_g: 24.8851 - mae__dry__total_g: 43.1714 - mae_all: 23.5630 - mae_gdm_g: 31.5166 - val_loss: 366.0359 - val_mae__dry__clover_g: 7.4941 - val_mae__dry__dead_g: 9.8274 - val_mae__dry__green_g: 27.8342 - val_mae__dry__total_g: 43.5001 - val_mae_all: 24.8297 - val_mae_gdm_g: 35.4927 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 751ms/step - loss: 302.0152 - mae__dry__clover_g: 6.2172 - mae__dry__dead_g: 11.8364 - mae__dry__green_g: 24.3093 - mae__dry__total_g: 40.4482 - mae_all: 22.7606 - mae_gdm_g: 30.9919 - val_loss: 316.7690 - val_mae__dry__clover_g: 7.4399 - val_mae__dry__dead_g: 9.6280 - val_mae__dry__green_g: 26.7679 - val_mae__dry__total_g: 39.1309 - val_mae_all: 23.4870 - val_mae_gdm_g: 34.4686 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 748ms/step - loss: 266.7889 - mae__dry__clover_g: 6.2182 - mae__d

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 750ms/step - loss: 343.4329 - mae__dry__clover_g: 6.3350 - mae__dry__dead_g: 11.3511 - mae__dry__green_g: 26.0833 - mae__dry__total_g: 43.4269 - mae_all: 23.9973 - mae_gdm_g: 32.7904 - val_loss: 308.3741 - val_mae__dry__clover_g: 6.9700 - val_mae__dry__dead_g: 12.8327 - val_mae__dry__green_g: 22.7037 - val_mae__dry__total_g: 41.0430 - val_mae_all: 22.9929 - val_mae_gdm_g: 31.4150 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 317.2516 - mae__dry__clover_g: 6.2785 - mae__dry__dead_g: 11.4232 - mae__dry__green_g: 25.5570 - mae__dry__total_g: 40.5423 - mae_all: 23.2815 - mae_gdm_g: 32.6066 - val_loss: 255.1552 - val_mae__dry__clover_g: 6.9056 - val_mae__dry__dead_g: 13.0212 - val_mae__dry__green_g: 20.9725 - val_mae__dry__total_g: 34.4899 - val_mae_all: 21.3635 - val_mae_gdm_g: 31.4281 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 288.9284 - mae__dry__clover_g: 6.3437 - mae_

/Users/jiaweilong/Downloads/advMaths/final/venv_py3.11_tensorflow/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 750ms/step - loss: 343.8331 - mae__dry__clover_g: 6.7597 - mae__dry__dead_g: 11.3297 - mae__dry__green_g: 25.6999 - mae__dry__total_g: 43.4428 - mae_all: 23.9049 - mae_gdm_g: 32.2923 - val_loss: 329.6394 - val_mae__dry__clover_g: 5.4135 - val_mae__dry__dead_g: 10.6413 - val_mae__dry__green_g: 27.5342 - val_mae__dry__total_g: 41.1744 - val_mae_all: 23.4742 - val_mae_gdm_g: 32.6076 - learning_rate: 1.0000e-04
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 311.6767 - mae__dry__clover_g: 6.8006 - mae__dry__dead_g: 10.9943 - mae__dry__green_g: 25.6294 - mae__dry__total_g: 40.3599 - mae_all: 23.1426 - mae_gdm_g: 31.9287 - val_loss: 276.5795 - val_mae__dry__clover_g: 5.4041 - val_mae__dry__dead_g: 9.9934 - val_mae__dry__green_g: 27.4794 - val_mae__dry__total_g: 35.5122 - val_mae_all: 22.0401 - val_mae_gdm_g: 31.8114 - learning_rate: 1.0000e-04
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 745ms/step - loss: 277.8702 - mae__dry__clover_g: 6.8005 - mae__

===== [CNN Baseline - gram] CV summary ===== 15 epochs, 3 target
val_mae_all per fold: [10.121 11.914 10.549 10.01  10.141]
mean ± std = 10.54708251953125 ± 0.7075833822485404
weighted_r2 per fold: [ 0.304 -0.171  0.498  0.39   0.33 ]
basline_gram mean R2 = 0.2702052972636771

===== [CNN Baseline - gram] CV summary ===== 30 epochs, 5 target
val_mae_all per fold: [11.332 14.766 12.259 16.392 12.652]
mean ± std = 13.479974555969239 ± 1.8397128198302635
weighted_r2 per fold: [ 0.426  0.287  0.595 -0.147  0.351]
basline_gram mean R2 = 0.30244747404764405

===== [Student Distill] 5-fold CV summary ===== 15 epochs, 3 target
MAE per fold: [ 9.151 10.531 10.454  9.288  9.149]
R2  per fold: [0.383 0.414 0.353 0.48  0.231]
mean R2 = 0.37238640442242643

===== [Student Distill] 5-fold CV summary ===== 30 epochs, 5 target
MAE per fold: [11.306 12.196 12.005 12.301 12.586]
R2  per fold: [0.469 0.419 0.64  0.395 0.362]
mean R2 = 0.4569973935200549

===== [Student Distill] 5-fold CV summary ===== (α=0.2, τ=1.0), 15 epochs, 3 target
MAE per fold: [ 9.151 10.531 10.454  9.288  9.149]
R2  per fold: [0.383 0.414 0.353 0.48  0.231]
mean R2 = 0.37238640442242643

===== [Student Distill] 5-fold CV summary ===== (α=0.2, τ=1.0), 30 epochs, 5 target
MAE per fold: [11.285 12.204 12.007 12.213 12.611]
R2  per fold: [0.475 0.417 0.638 0.401 0.354]
mean R2 = 0.4568422577684272

===== [Student Distill] 5-fold CV summary ===== (α=0.5, τ=1.0), 15 epochs, 3 target
MAE per fold: [ 9.121  9.743 10.556  9.187  8.893]
R2  per fold: [0.457 0.428 0.421 0.537 0.356]
mean R2 = 0.43976178722121323

===== [Student Distill] 5-fold CV summary ===== (α=0.6, τ=1.0), 30 epochs, 5 target
MAE per fold: [11.199 12.325 11.998 11.8   12.604]
R2  per fold: [0.467 0.404 0.634 0.445 0.352]
mean R2 = 0.460478188988293

===== [Student Distill] 5-fold CV summary ===== (α=0.4, τ=1.0), 30 epochs, 5 target
MAE per fold: [11.238 12.348 12.005 12.191 12.555]
R2  per fold: [0.472 0.403 0.632 0.399 0.351]
mean R2 = 0.4513830060100158

===== [Student Distill] 5-fold CV summary ===== (α=0.5, τ=1.0), 30 epochs, 5 target ---> Second!!
MAE per fold: [11.21  12.336 11.941 11.874 12.572]
R2  per fold: [0.471 0.399 0.642 0.445 0.353]
mean R2 = 0.46223788567622853

===== [Student Distill] 5-fold CV summary ===== (α=0.55, τ=1.0), 30 epochs, 5 target ---> Best!!
MAE per fold: [11.205 12.328 11.984 11.811 12.591]
R2  per fold: [0.469 0.405 0.635 0.451 0.354]
mean R2 = 0.4627702221111635



In [63]:

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# Define data for the conclusion
models = [
    'Mean Baseline,',
    'CNN Baseline, 30 epochs, ~49mins',
    'Teacher Model, 30 epchs, ~65mins',
    'Student (Fine-tuned only), 30 epochs, ~73mins',
    'Distilled Student (α=0.5, τ=1.0), 30 epochs, ~73mins',   
    'Distilled Student (α=0.55, τ=1.0), 30 epochs, ~72mins',   
]

inputs = [
    'N/A',
    'Image Only',
    'Image + Metadata',
    'Image Only',
    'Image Only',
    'Image Only',
]

# Updated R² values from your notebook results
r2_scores = [
    0.243,
    0.302,
    0.740,
    0.457,
    0.462,
    0.463,
]

insights = [
]

# ============== Print Summary Statistics ==============
print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)

for i, model in enumerate(models):
    print(f"\n{model}")
    print(f"  Inputs: {inputs[i]}")
    print(f"  Mean Weighted R²: {r2_scores[i]:.3f}")
    # print(f"  Insight: {insights[i]}")

print("\n" + "="*60)
print("KEY FINDINGS:")
print("="*60)
print(f"• Improvement from Mean Baseline → Best Distilled Student: "
      f"{((r2_scores[4] - r2_scores[0]) / r2_scores[0] * 100):.1f}%")
print(f"• Performance gap (Best Student vs Teacher): "
      f"{(r2_scores[2] - r2_scores[4]):.3f}")
print(f"• Best α=0.55 outperforms α=0.5 by: "
      f"{((r2_scores[5] - r2_scores[4]) / r2_scores[4] * 100):.1f}%")




MODEL PERFORMANCE SUMMARY

Mean Baseline,
  Inputs: N/A
  Mean Weighted R²: 0.243

CNN Baseline, 30 epochs, ~49mins
  Inputs: Image Only
  Mean Weighted R²: 0.302

Teacher Model, 30 epchs, ~65mins
  Inputs: Image + Metadata
  Mean Weighted R²: 0.740

Student (Fine-tuned only), 30 epochs, ~73mins
  Inputs: Image Only
  Mean Weighted R²: 0.457

Distilled Student (α=0.5, τ=1.0), 30 epochs, ~73mins
  Inputs: Image Only
  Mean Weighted R²: 0.462

Distilled Student (α=0.55, τ=1.0), 30 epochs, ~72mins
  Inputs: Image Only
  Mean Weighted R²: 0.463

KEY FINDINGS:
• Improvement from Mean Baseline → Best Distilled Student: 90.1%
• Performance gap (Best Student vs Teacher): 0.278
• Best α=0.55 outperforms α=0.5 by: 0.2%
